## Describe your model -> fine-tuned Collective Cognition
Dataset for training must be placed in the dataset directory

# Install necessary libraries

In [1]:
!pip install torch torchvision torchaudio #--index-url https://download.pytorch.org/whl/cu124


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install -q -U --no-build-isolation peft bitsandbytes  trl   accelerate  transformers accelerate bitsandbytes datasets loralib sentencepiece  einops modelz-llm huggingface_hub

#!pip install flash_attn xformers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoProcessor,
)
from transformers import GenerationConfig, pipeline

In [4]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

In [5]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
from huggingface_hub import notebook_login

notebook_login()

In [6]:
!rm -rf sample_data
!rm -rf results


# Define Hyperparameters

In [7]:
model_name = "meta-llama/Llama-3.2-1B"  # "openchat/openchat_3.5"  # "teknium/CollectiveCognition-v1.1-Mistral-7B" # use this if you have access to the official LLaMA 2 model "meta-llama/Llama-2-7b-chat-hf", though keep in mind you'll need to pass a Hugging Face key argument
dataset_name = "./dataset"
# dataset_name = "./dataset/"
new_model = "joetib/en-twi-70m"
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1
use_4bit = False
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False
output_dir = "./results"
num_train_epochs = 3
fp16 = False
bf16 = False
per_device_train_batch_size = 3
per_device_eval_batch_size = 2
gradient_accumulation_steps = 2
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "constant"
max_steps = -1
warmup_ratio = 0.03
group_by_length = True
save_steps = 100
save_total_limit = 2
logging_steps = 4
max_seq_length = 512
packing = False
device_map = "mps"

In [8]:
!export PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0

#Load Datasets and Train

#Run Inference

In [9]:
!rm -rf results

In [10]:
# Load datasets
train_dataset = load_dataset(dataset_name, split="train[:1000]")

# test_dataset = load_dataset("dataset", split="test")

# Preprocess datasets


train_dataset_mapped = train_dataset.map(
    lambda examples: {
        "text": [
            "<s>Translate to Twi[INST]"
            + prompt.strip()
            + "[/INST]"
            + response.strip()
            + "</s>"
            for prompt, response in zip(examples["english"], examples["twi"])
        ]
    },
    batched=True,
)
# valid_dataset_mapped = test_dataset.map(lambda examples: {'text': [f'<s>' + prompt + '</s>' + response for prompt, response in zip(examples['query'], examples['response'])]}, batched=True)

In [11]:
train_dataset_mapped[0]

{'english': ' Oh Jehovah Keep My Young Girl Faithful ! ',
 'twi': ' Oo Yehowa Boa Me Babea Kumaa Yi Ma Onni Nokware ! ',
 'text': '<s>Translate to Twi[INST]Oh Jehovah Keep My Young Girl Faithful ![/INST]Oo Yehowa Boa Me Babea Kumaa Yi Ma Onni Nokware !</s>'}

In [12]:
len(train_dataset_mapped)

5000

In [13]:
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map=device_map,
    # attn_implementation="flash_attention_2",  # only large context
    # trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
tokenizer = AutoTokenizer.from_pretrained(model_name, add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# just for 128k context
# tokenizer.padding_side="left"

In [14]:
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    # target_modules=[
    #     "q_proj",
    #     "k_proj",
    #     "v_proj",
    #     "o_proj",
    #     "gate_proj",
    #     "up_proj",
    #     "down_proj",
    # ],
)
# Set training parameters
training_arguments = SFTConfig(
    max_seq_length=max_seq_length,
    dataset_text_field="text",
    packing=packing,
    use_mps_device=True,
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    # optim=optim,
    # use_mps_device=True,
    save_steps=save_steps,
    # save_total_limit=save_total_limit,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    # max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    # lr_scheduler_type=lr_scheduler_type,
    report_to="all",
    # evaluation_strategy="steps",
    # eval_steps=5,  # Evaluate every 20 steps
)

# Set supervised fine-tuning parameters

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_mapped,
    # eval_dataset=valid_dataset_mapped,  # Pass validation dataset here
    peft_config=peft_config,
    args=training_arguments,
)

/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/transformers/training_args.py:2214: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.
/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/codecarbon/input.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
[codecarbon INFO @ 17:48:00] [setup] RAM Tracking...
[codecarbon INFO @ 17:48:00] [setup] GPU Tracking...
[codecarbon INFO @ 17:48:00] No GPU found.
[codecarbon INFO @ 17:48:00] [setup] CPU Tracking...
[codecarbon WARNING @ 17:48:00] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 17:48:01] CPU Model on constant consumption mode: Apple M1 Pro

In [ ]:
trainer.train()
trainer.model.save_pretrained(new_model)

/Users/joetib/Desktop/projects/graduation/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
[codecarbon INFO @ 17:48:18] Energy consumed for RAM : 0.000025 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:48:18] Energy consumed for all CPUs : 0.000021 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 17:48:18] 0.000046 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:48:33] Energy consumed for RAM : 0.000050 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:48:33] Energy consumed for all CPUs : 0.000042 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 17:48:33] 0.000092 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:48:48] Energy consumed for RAM : 0.000075 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:48:48] Energy consumed for all CPUs : 0.000063 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 1

Step,Training Loss
4,3.807400
8,3.958500
12,4.064500
16,4.098900
20,4.113400
24,4.237500


[codecarbon INFO @ 17:58:03] Energy consumed for RAM : 0.000998 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:58:03] Energy consumed for all CPUs : 0.000834 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 17:58:03] 0.001832 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:58:18] Energy consumed for RAM : 0.001023 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:58:18] Energy consumed for all CPUs : 0.000855 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 17:58:18] 0.001878 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:58:33] Energy consumed for RAM : 0.001048 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:58:33] Energy consumed for all CPUs : 0.000876 kWh. Total CPU Power : 5.0 W
[codecarbon INFO @ 17:58:33] 0.001924 kWh of electricity used since the beginning.
[codecarbon INFO @ 17:58:48] Energy consumed for RAM : 0.001073 kWh. RAM Power : 6.0 W
[codecarbon INFO @ 17:58:48] Energy consumed for all CPUs : 0.000896 kWh. Total CPU Power : 5.0 W
[codecarbon

In [15]:
del model
del trainer
del peft_config
del tokenizer

In [14]:
del pipeline

In [16]:
import gc

gc.collect()
torch.cuda.empty_cache()
del torch
gc.collect()
import torch

gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

In [19]:
import torch

torch.mps.empty_cache()

In [20]:
!rm -rf results

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


#Merge the model and store in Google Drive

In [21]:
# Merge and save the fine-tuned model
# from google.colab import drive
# drive.mount('/content/drive')

# model_path = "/content/drive/MyDrive/ibl-tutoring-mistral-7b-v1.1"  # change to your preferred path

# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Save the merged model

In [22]:
from transformers import pipeline

p = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use mps


In [23]:
train_dataset_mapped[0]

{'english': ' Oh Jehovah Keep My Young Girl Faithful ! ',
 'twi': ' Oo Yehowa Boa Me Babea Kumaa Yi Ma Onni Nokware ! ',
 'text': '<s>Translate to Twi[INST]Oh Jehovah Keep My Young Girl Faithful ![/INST]Oo Yehowa Boa Me Babea Kumaa Yi Ma Onni Nokware !</s>'}

In [25]:
p(
    "<s>Translate to Twi[INST]how are you? ![/INST]",
    max_new_tokens=20,
    do_sample=True,
    temperature=1,
)

[{'generated_text': '<s>Translate to Twi[INST]how are you? ![/INST]hɛ awoɔ?</s>'}]

In [ ]:
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

In [ ]:
model.push_to_hub(repo_id="Joetib/" + new_model, max_shard_size="2GB")
tokenizer.push_to_hub(repo_id="Joetib/" + new_model)

In [ ]:
!rm -rf ibl-fordham-7b-mistral
!rm -rf results
!rm -rf pretrained

In [ ]:
!rm -rf ibl-llm-tutoring-7b/

In [ ]:
from transformers import pipeline

p = pipeline("text-generation", model=model, tokenizer=tokenizer)
p("<s>[INST] describe fordham university? [/INST] ")

In [ ]:
print(
    p("<s>[INST]Photosynthesis[/INST]", max_new_tokens=512, temperature=0.7)[0][
        "generated_text"
    ]
)

In [ ]:
print(p("<s>[INST]Information Theory[/INST]", max_new_tokens=512)[0]["generated_text"])

## Clear GPU

In [ ]:
import gc

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
del torch
gc.collect()
import torch

gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

In [ ]:
del base_model
gc.collect()
torch.cuda.empty_cache()

# convert to gguf

In [ ]:
from huggingface_hub import snapshot_download

model_id = "ibleducation/" + new_model
snapshot_download(
    repo_id=model_id, local_dir="initial", local_dir_use_symlinks=False, revision="main"
)

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp.git

In [ ]:
!pip install -r llama.cpp/requirements.txt

In [ ]:
model_id

In [ ]:
!python llama.cpp/convert.py initial \
  --outfile ibl-fordham-7b.gguf \
  --vocab-dir initial \
  --outtype q8_0

In [ ]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HF_TOKEN_HERE"
os.environ["HUGGING_FACE_HUB_TOKEN"] = "YOUR_HF_TOKEN_HERE"

In [ ]:
!export HUGGING_FACE_HUB_TOKEN=YOUR_HF_TOKEN_HERE

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
new_model_id = model_id + ".gguf"
api.create_repo(new_model_id, exist_ok=True, repo_type="model")
api.upload_file(
    path_or_fileobj="ibl-fordham-7b.gguf",
    path_in_repo="ibl-fordham-7b-mistral.gguf",
    repo_id=new_model_id,
)

# Load a fine-tuned model from Drive and run inference

In [ ]:
!pip install ctransformers langchain

In [ ]:
from langchain.llms import CTransformers

In [ ]:
hf_original = CTransformers(
    model="./ibl-fordham-7b.gguf", model_type="mistral", gpu_layers=50
)

In [ ]:
hf_original(
    "<s>[INST]What is the date range for the academic calendar provided?</INST>"
)

In [ ]:
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

drive.mount("/content/drive")

model_path = "/content/drive/MyDrive/ibl-tutoring-mistral-7b-v1.1"  # change to the path where your model is saved

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
from transformers import pipeline

prompt = "What is 2 + 2?"  # change to your desired prompt
gen = pipeline("text-generation", model=model, tokenizer=tokenizer)
result = gen(prompt)
print(result[0]["generated_text"])

In [ ]:
import os

In [ ]:
# @markdown ---
# @markdown #### Project Config
# @markdown Note: if you are using a restricted/private model, you need to enter your Hugging Face token in the next step.
project_name = "en-pythia-410m"  # @param {type:"string"}
model_name = "EleutherAI/pythia-410m-deduped-v0"  # @param {type:"string"}

# @markdown ---
# @markdown #### Push to Hub?
# @markdown Use these only if you want to push your trained model to a private repo in your Hugging Face Account
# @markdown If you dont use these, the model will be saved in Google Colab and you are required to download it manually.
# @markdown Please enter your Hugging Face write token. The trained model will be saved to your Hugging Face account.
# @markdown You can find your token here: https://huggingface.co/settings/tokens
push_to_hub = True  # @param ["False", "True"] {type:"raw"}
hf_token = "YOUR_HF_TOKEN_HERE"  # @param {type:"string"}
hf_username = "Joetib"  # @param {type:"string"}
new_model = "en-twi-410m"  # @param {type:"string"}
# @markdown ---
# @markdown #### Hyperparameters
unsloth = False  # @param ["False", "True"] {type:"raw"}
learning_rate = 2e-4  # @param {type:"number"}
num_epochs = 5  # @param {type:"number"}
batch_size = 1  # @param {type:"slider", min:1, max:32, step:1}
block_size = 1024  # @param {type:"number"}
trainer = "sft"  # @param ["generic", "sft"] {type:"string"}
warmup_ratio = 0.1  # @param {type:"number"}
weight_decay = 0.01  # @param {type:"number"}
gradient_accumulation = 4  # @param {type:"number"}
mixed_precision = "none"  # @param ["fp16", "bf16", "none"] {type:"string"}
peft = True  # @param ["False", "True"] {type:"raw"}
quantization = "none"  # @param ["int4", "int8", "none"] {type:"string"}
lora_r = 16  # @param {type:"number"}
lora_alpha = 32  # @param {type:"number"}
lora_dropout = 0.05  # @param {type:"number"}

os.environ["HF_TOKEN"] = hf_token
os.environ["HF_USERNAME"] = hf_username

conf = f"""
task: llm-{trainer}
base_model: {model_name}
project_name: {project_name}
log: tensorboard
backend: local

data:
  path: ./cleaned-dataset/
  train_split: train
  valid_split: null
  chat_template: null
  column_mapping:
    text_column: text

params:
  block_size: {block_size}
  lr: {learning_rate}
  warmup_ratio: {warmup_ratio}
  weight_decay: {weight_decay}
  epochs: {num_epochs}
  batch_size: {batch_size}
  gradient_accumulation: {gradient_accumulation}
  mixed_precision: {mixed_precision}
  peft: {peft}
  quantization: {quantization}
  lora_r: {lora_r}
  lora_alpha: {lora_alpha}
  lora_dropout: {lora_dropout}
  unsloth: {unsloth}

hub:
  username: ${{HF_USERNAME}}
  token: ${{HF_TOKEN}}
  push_to_hub: {push_to_hub}
  repo_id: {new_model}
"""

with open("conf.yaml", "w") as f:
    f.write(conf)

In [ ]:
!autotrain --config conf.yaml